In [7]:
# Alright, second time is the charm for this freaking thing

from pathlib import Path
import polars as pl

# run expectancy matrix
REM = Path("../../data_pipeline/re_matrices/re288_matrix.parquet")
rem = pl.read_parquet(REM)
rem = rem.filter((pl.col("Level") == "D1") & (pl.col("year") == "2026"))

DATA_DIR = Path("../../data_pipeline/wbaserunners/D1/2026")
# We are going to filter for only the features that we need
# Humans use SpinAxis to help determine pitch type, but GMM exploits spin axis
lf = (
    pl.scan_parquet(str(DATA_DIR / "**/*.parquet"))
    .select([
        "Pitcher", "PitcherThrows",
        "TaggedPitchType", "RelSpeed", 
        "SpinRate", "RelHeight",
        "InducedVertBreak", "HorzBreak", "Extension"
    ])
    .drop_nulls()
    .filter(
        ~pl.col("PitcherThrows").is_in(["Both", "Undefined"])
    )
    .filter(
        ~pl.col("TaggedPitchType").is_in(["Knuckleball", "Other", "Undefined"])
    )
)
df = lf.collect()
df.shape

(1905756, 9)

In [8]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

pl.Config.set_tbl_rows(-1)
print(df["TaggedPitchType"].value_counts())

shape: (11, 2)
┌──────────────────┬────────┐
│ TaggedPitchType  ┆ count  │
│ ---              ┆ ---    │
│ str              ┆ u32    │
╞══════════════════╪════════╡
│ Fastball         ┆ 761259 │
│ ChangeUp         ┆ 212631 │
│ Slider           ┆ 390460 │
│ TwoSeamFastBall  ┆ 17478  │
│ OneSeamFastBall  ┆ 183    │
│ FourSeamFastBall ┆ 94171  │
│ Sweeper          ┆ 16782  │
│ Cutter           ┆ 126996 │
│ Sinker           ┆ 127875 │
│ Splitter         ┆ 13396  │
│ Curveball        ┆ 144525 │
└──────────────────┴────────┘


In [9]:
### We'll start with the fastball model first
import numpy as np
### Feature engineering

## Anchoring Fastballs

# Spin Axis to x, y coords and reflecting for lefty pitchers
df = df.with_columns(
    HorzBreak=(pl.when(pl.col("PitcherThrows") == "Left")
            .then(-pl.col("HorzBreak"))
            .otherwise(pl.col("HorzBreak")))
    )

# min_pitches
min_pitches = 300
df = df.filter(
    pl.len().over("Pitcher") >= min_pitches
)

In [13]:
# Mason Edwards test (really interested to see because it seems like he has 3 pitch mix with a slurve that takes slider shape too)
govel = df.filter(
        pl.col("Pitcher").is_in(["Govel, Grant"])
    ).select([
        "RelSpeed", "SpinRate",
        "InducedVertBreak", "HorzBreak", 
        "Extension", "RelHeight" 
    ])
# Gabe Gaeckle test (seems to have 4 distinct pitches)
gaeckle = df.filter(
        pl.col("Pitcher").is_in(["Gaeckle, Gabe"])
    ).select([
        "RelSpeed", "SpinRate", "RelHeight", "InducedVertBreak", "HorzBreak", "Extension"
    ])

test_dfs = {
    "govel":govel, 
    "gaeckle":gaeckle
}

In [14]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClusterMixin
import numpy as np
import pandas as pd

results = {}

for name, pdf in test_dfs.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(pdf)

    best_bic = np.inf
    best_model = None
    best_k = None
    
    bic_scores = []
    for k in range(1,7):
        gmm = GaussianMixture(
            n_components=k,
            covariance_type="full",
            n_init=20,
            random_state=42
        )

        gmm.fit(X_scaled)
        bic = gmm.bic(X_scaled)

        # adding icl alongside bic to choose more defined clusters
        resp = gmm.predict_proba(X_scaled)             
        entropy = -np.sum(resp * np.log(resp + 1e-12))  
        icl = bic + 2 * entropy 

        bic_scores.append((k, bic, icl))

        # lower icl takes precedent over bic
        if icl < best_bic:
            best_bic = icl
            best_model = gmm
            best_k = k

    labels = best_model.predict(X_scaled)

    results[name] = {
        "best_k": best_k,
        "best_bic": best_bic,
        "bic_table": pd.DataFrame(bic_scores, columns=["k", "BIC", "ICL"]),
        "labels": labels,
        "model": best_model,
        "scaler": scaler
    }

    print(f"\n{name}\n Best k = {best_k}")
    print(results[name]["bic_table"])


govel
 Best k = 4
   k           BIC           ICL
0  1  19454.987891  19454.987891
1  2  16020.639143  16052.428706
2  3  14568.037618  14595.654266
3  4  14311.770276  14363.895355
4  5  14209.977305  14439.747269
5  6  14181.342641  14517.086074

gaeckle
 Best k = 4
   k           BIC           ICL
0  1  12546.335441  12546.335441
1  2   9003.971421   9003.974296
2  3   8002.940931   8002.941663
3  4   7840.348167   7875.480961
4  5   7826.091283   8072.480750
5  6   7907.600111   8448.871369


In [12]:
# To later interpret in original 
for name, result in results.items():
    model = result["model"]
    scaler = result["scaler"]
    labels = result["labels"]

    cluster_means = scaler.inverse_transform(model.means_)

    feature_names = test_dfs[name].columns

    cluster_df = pd.DataFrame(
        cluster_means,
        columns=feature_names
    )
    cluster_df["Count"] = np.bincount(labels)
    cluster_df.index.name = "Cluster"
    print(f"\n{name}")
    print(f"Best k = {result['best_k']}")
    print(cluster_df.round(2))


edwards
Best k = 3
         RelSpeed  SpinRate  InducedVertBreak  HorzBreak  Extension  RelHeight  Count
Cluster                                                                              
0           91.93   2319.19             20.85       7.79       6.06       6.45    750
1           80.66   2472.76            -10.25     -10.75       5.54       6.16    654
2           82.51   1795.87             13.47      14.35       6.08       6.18    112

gaeckle
Best k = 4
         RelSpeed  SpinRate  RelHeight  InducedVertBreak  HorzBreak  Extension  Count
Cluster                                                                              
0           81.38   2687.82       5.32            -10.52     -10.52       5.46    246
1           94.34   2469.17       5.24             17.23       8.25       6.27    457
2           85.38   2653.91       5.16             -1.70      -8.08       5.73    329
3           88.55   1901.53       4.99              5.00      16.59       6.29    158
